In [2]:
import pandas as pd
df = pd.read_csv("../data/raw/gender.csv")

In [4]:
import re
from collections import Counter

In [3]:
d = df

In [5]:
pattern1 = r"\b(?:I(?:'m|'m a| am| am a| identify as| identify as a)\s)(male|female|father|mother|brother|sister|daughter|son|man|women|guy|girl|boy|husband|wife)\b"  # For self-references
pattern2 = r"\b(\d{2})([MF])\b|\b([MF])(\d{2})\b"

# Initialize a list to store all matches
all_matches = []

# Extract matches from the 'post' column
for post in df['post']:
    matches = re.findall(pattern1, post)  # Self-reference matches
    matches += re.findall(pattern2, post)  # Age/gender matches
    
    all_matches.extend(matches)

# Count occurrences of each match
match_counts = Counter(all_matches)

# Convert to DataFrame for display
results_df = pd.DataFrame(match_counts.items(), columns=['Pattern', 'Count']).sort_values(by='Count', ascending=False)

# Display the table
print(results_df)

# Save to CSV
results_df.to_csv("../data/matches_output.csv", index=False)

         Pattern  Count
138  (44, F, , )    708
62           guy    292
37   (25, M, , )    278
241  (51, F, , )    257
7    (27, F, , )    236
..           ...    ...
282  (, , M, 92)      1
285  (, , M, 74)      1
286  (, , M, 66)      1
287  (74, F, , )      1
304  (, , F, 95)      1

[305 rows x 2 columns]


In [6]:
# Print the sum of all matches
total_matches = sum(match_counts.values())
print(f"Total number of matches: {total_matches}")

Total number of matches: 10031


In [10]:
results_df.to_csv("../data/pattern_counts.csv", index=False)

In [20]:
pattern = r"\b(?:\d{2}[MF]|[MF]\d{2}|\([MF]\d{2}\))\b"
count = d['post'].str.count(pattern).sum()

# Iterate through the rows and extract sentences containing matches
for index, post in enumerate(d['post']):
    sentences = re.split(r'(?<=[.!?])\s+', post)

    for sentence in sentences:
        matches = re.findall(pattern, sentence)
        if matches:
            print(f"Post {index + 1}: {sentence}")
            print(f"Matches: {matches}")
            print("-" * 50)

Post 11: I mean I am aware that French words spoken in an Irish accent is probably painful for French people to hear Speaking from experience as a 36F, men in their late 30s / early 40s aren’t much better 😩 He won the battle, but lost the war TikTok can fuck right off Huh, who knew religion would actually pay off Yes to no.3!
Matches: ['36F']
--------------------------------------------------
Post 26: 42F here…buying Christmas presents for my teens.
Matches: ['42F']
--------------------------------------------------
Post 26: I (42F) literally starved myself for years and went on crazy new fab diets, even did a mommy makeover BBL360 because he (40M) used to tell me how fat I was going to get if I get out of the military only for him to sleep with a chunky woman (43F) who I decided to call fridge (since she is all square, has no shape, definitely a pancake butt, and no lips).
Matches: ['42F', '40M', '43F']
--------------------------------------------------
Post 26: But we have to be civi

In [24]:
# Define the regex pattern for self-references
pattern = r"\b(?:I(?:'m| am)|I'm a|I am a)\s(?:male|female|father|mother|brother|sister)\b"

# Count total matches
count = d['post'].str.count(pattern).sum()
print(f"Total matches: {count}")

# Iterate through the rows and extract sentences containing matches
for index, post in enumerate(d['post']):
    sentences = re.split(r'(?<=[.!?])\s+', post)

    for sentence in sentences:
        matches = re.findall(pattern, sentence)
        if matches:
            print(f"Post {index + 1}: {sentence}")
            print(f"Matches: {matches}")
            print("-" * 50)

Total matches: 5
Post 29: For the record I'm female and childless.
Matches: ["I'm female"]
--------------------------------------------------
Post 29: LOL so I'm female and childless but I genuinely wonder about this argument too!
Matches: ["I'm female"]
--------------------------------------------------
Post 105: The problem is that I grew up on a small city in Northern Canada in the 90's, and I'm female, so I don't have all the standard symptoms.
Matches: ["I'm female"]
--------------------------------------------------
Post 308: I'm a mother of 2, I carried them for 9 months I labored in a hospital bed and birthed them and I breastfed both of them, day and night for a year straight not my husband, ME.
Matches: ["I'm a mother"]
--------------------------------------------------
Post 900: Umm, I am female, my mom is female, my two sisters are female, and all three of us girls breast-fed, watched my mom breast-feed each other, and frequently showered with our mom as small children.
Mat

In [9]:
# Define the regex patterns
pattern1 = r"\b(?:I(?:'m| am| am a|I'm a)\s)(male|female|father|mother|brother|sister)\b"  # For self-references
pattern2 = r"\b(\d{2})([MF])\b"  # For 22M or 22F

# Function to replace gender references in a post
def replace_gender(post):
    # Replace self-references (e.g., "I am a male" -> "I am a human")
    post = re.sub(pattern1, lambda m: re.sub(r"\b(male|female|father|mother|brother|sister)\b", "human", m.group(0)), post)
    
    # Replace "22M" or "22F" with just the number (e.g., "22M" -> "22" and "22F" -> "22")
    post = re.sub(pattern2, lambda m: m.group(1), post)  # Remove the 'M' or 'F' and keep the number
    
    return post

# Apply the replacement function to each post in the DataFrame
d['post_cleaned'] = d['post'].apply(replace_gender)

pattern_test = r"\b(?:I(?:'m| am| am a|I'm a)\s)(human)\b"
# Print all sentences within post that contain pattern_test
for index, post in enumerate(d['post_cleaned']):
    sentences = re.split(r'(?<=[.!?])\s+', post)

    for sentence in sentences:
        matches = re.findall(pattern_test, sentence)
        if matches:
            print(f"Post {index + 1}: {sentence}")
            print(f"Matches: {matches}")
            print("-" * 50)


Post 5: None of the anointing touched me inappropriately and it was done by old ladies (I am person).
Matches: ['person']
--------------------------------------------------
Post 372: I'm person, 5'4", started at about 150, now I'm at 133.
Matches: ['person']
--------------------------------------------------
Post 713: I am a person for my kids who is save and positive.
Matches: ['person']
--------------------------------------------------
Post 764: I didn’t choose cancer, so I am a person who had cancer.
Matches: ['person']
--------------------------------------------------
Post 863: For context, I'm person.
Matches: ['person']
--------------------------------------------------
Post 897: I'm person, I'm old, my husband wasn't exactly a hands on dad when my kids were small.
Matches: ['person']
--------------------------------------------------


In [2]:
from transformers import DistilBertTokenizer

# Initialize tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Example token IDs (with leading zeros for demonstration)
token_ids = [101, 2190, 2126, 2003, 2131, 2014, 2000, 2079, 2009, 1012, 2442, 2022, 2037, 6280, 5030, 5315, 2011, 2085, 1012, 2053, 2017, 5807, 1005, 1056, 1012, 1999, 3558, 2998, 2477, 2131, 9157, 1012, 2009, 1005, 1055, 2112, 1998, 20463, 1997, 2009, 1012, 2002, 3791, 2000, 5138, 2008, 1008, 2374, 2336, 2031, 16160, 1012, 2115, 2684, 2097, 4060, 2039, 2006, 2009, 4312, 1998, 2113, 2035, 2003, 2025, 2157, 2061, 2488, 2000, 2022, 7481, 2055, 2673, 1012, 5160, 2039, 1012, 2034, 2088, 3471, 2123, 1005, 1056, 11224, 1996, 3626, 1012, 2009, 1005, 2222, 2203, 1999, 4000, 1012, 2017, 1005, 2128, 2058, 16416, 11873, 1012, 2017, 2064, 2593, 2486, 1996, 3277, 2011, 3331, 2000, 2014, 5667, 2055, 2009, 1010, 2065, 2016, 2180, 1005, 1056, 2552, 24202, 2074, 4468, 2108, 2105, 1996, 2336, 1012, 1999, 2019, 7812, 2088, 1037, 7132, 11112, 1998, 1037, 7132, 6805, 1999, 2709, 2052, 2393, 1996, 10527, 3305, 2049, 2025, 7929, 2021, 2057, 2053, 2936, 2444, 1999, 2008, 2088, 1012, 1045, 6655, 2017, 1005, 2128, 4569, 2012, 4243, 1012, 2123, 1005, 1056, 4521, 1996, 3756, 4586, 1012, 2065, 1045, 2001, 2115, 2564, 1045, 1005, 1040, 2763, 2681, 2017, 2005, 2025, 2478, 20423, 2015, 8840, 2140, 1012, 2519, 2055, 3438, 27487, 4237, 1010, 3456, 3621, 6260, 1998, 2067, 9194, 2830, 1037, 2210, 1012, 2008, 1005, 1055, 1037, 4121, 11679, 1012, 2016, 2003, 2074, 2667, 2000, 4906, 1999, 2007, 1037, 2047, 2155, 21461, 2015, 1012, 16028, 2788, 2007, 16931, 2015, 2008, 2502, 2021, 1045, 1005, 1049, 3241, 2672, 3577, 1010, 2148, 2789, 1012, 2023, 2038, 5717, 9185, 2006, 2017, 1012, 2057, 2024, 2035, 2367, 1012, 2043, 1045, 2001, 2115, 2287, 1045, 6283, 1996, 2126, 1045, 2246, 1998, 2018, 2053, 2969, 7023, 1012, 2085, 1999, 2026, 2753, 1005, 1055, 1045, 2298, 2012, 4620, 1997, 2402, 2033, 1998, 19148, 1045, 2001, 2941, 3492, 11519, 2559, 1012, 8307, 4873, 2003, 12802, 2055, 2017, 1998, 10261, 2027, 2020, 2007, 2017, 1012, 2008, 1005, 1055, 2074, 2129, 2166, 2003, 1012, 2111, 2066, 2023, 2611, 2174, 1010, 7539, 1997, 2054, 1996, 2648, 2003, 2066, 2024, 2000, 2022, 6770, 6340, 2005, 1996, 16021, 8586, 29366, 2027, 2031, 2503, 2008, 2191, 2068, 2552, 2023, 2126, 1012, 1045, 2052, 8406, 2769, 2006, 2009, 2008, 2016, 2038, 2521, 2062, 16021, 8586, 29366, 2084, 2017, 2412, 2097, 1998, 2515, 2023, 4485, 1999, 1037, 13727, 3535, 2000, 17894, 2014, 2814, 2061, 2027, 2097, 2066, 2014, 1012, 2404, 2009, 2041, 1997, 2115, 2568, 1012, 2115, 3433, 2323, 2022, 1000, 3398, 2035, 4364, 2215, 1037, 2611, 2040, 2987, 1005, 1056, 2360, 12873, 4485, 2066, 2008, 1000, 1012, 2279, 1012, 1005, 14163, 14735, 1012, 4875, 4485, 1012, 5674, 2129, 2116, 2335, 2111, 2024, 2397, 2008, 9297, 2968, 2000, 4339, 2023, 1012, 1059, 24475, 3047, 2000, 2014, 4190, 1029, 2310, 4313, 2078, 1047, 2595, 2015, 22038, 2243, 2278, 2546, 2615, 3501, 2595, 4859, 25708, 2736, 1037, 2204, 2051, 2000, 8970, 1996, 6061, 1997, 20302, 3348, 2046, 1996, 4512, 1012, 18140, 1012, 2017, 1005, 2310, 2042, 22673, 2006, 1012, 2987, 1005, 1056, 2812, 2296, 2060, 2711, 2003, 8073, 16789, 1012, 2079, 2025, 2425, 1037, 3969, 1012, 2065, 2027, 5060, 2115, 5938, 102] # Example input

# Convert to string (handle leading zeros if needed)
token_ids = [int(str(i)) for i in token_ids]  # Ensure all are integers

# Convert IDs to tokens
tokens = tokenizer.convert_ids_to_tokens(token_ids)

# Untokenize to get the final string
untokenized_text = tokenizer.convert_tokens_to_string(tokens)

print(untokenized_text)


[CLS] best way is get her to do it . must be their 9th wedding anniversary by now . no you shouldn ' t . in physical sports things get ripped . it ' s part and parcel of it . he needs to accept that * football children have instincts . your daughter will pick up on it anyway and know all is not right so better to be honest about everything . lawyer up . first world problems don ' t screw the crew . it ' ll end in tears . you ' re overreacting . you can either force the issue by talking to her seriously about it , if she won ' t act ghent just avoid being around the children . in an ideal world a gentle tap and a gentle bite in return would help the infant understand its not ok but we no longer live in that world . i bet you ' re fun at parties . don ' t eat the yellow snow . if i was your wife i ' d probably leave you for not using paragraphs lol . feet about 60cm apart , legs slightly bent and back arched forward a little . that ' s a huge leap . she is just trying to fit in with a ne

In [15]:
import re

def filter_contamination(post):
    # Define the regex patterns 
    pattern1 = r"\b(?:I(?:'m|'m a| am| am a| identify as| identify as a)\s)(male|female|father|mother|brother|sister|daughter|son|man|women|guy|girl|boy|husband|wife)\b"  # For self-references
    pattern2 = r"\b(\d{2})([MFmf])\b|\b([MFmf])(\d{2})\b"

    # Replace self-references (e.g., "I am a male" -> "I am a human")
    post = re.sub(pattern1, lambda m: re.sub(r"\b(male|female|father|mother|brother|sister)\b", "human", m.group(0)), post)
        
    # Replace "22M" or "22F" with just the number (e.g., "22M" -> "22" and "22F" -> "22")
    post = re.sub(pattern2, lambda m: m.group(1) if m.group(1) else m.group(4), post)

    matches = re.findall(pattern1, post) + re.findall(pattern2, post)
        
    # Log the last 50 characters of the post if contamination patterns are found
    if re.search(pattern1, post) or re.search(pattern2, post):
        print(f"Post contains contamination: {post[-50:]}")

    return post, matches if matches else None

# Hardcoded test case
test_post = "a a a a I'm f57 and living"
result = filter_contamination(test_post)
print(f"Transformed Post: {result[0]}")
print(f"Matches Found: {result[1]}")

Transformed Post: a a a a I'm 57 and living
Matches Found: None
